In [13]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName('etl') \
    .config("spark.jars", "/opt/spark/jars/iceberg-spark-runtime-3.5_2.12-1.6.0.jar") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.spark_catalog.type", "hive") \
    .config("spark.sql.catalog.local.warehouse", "s3a://datalake/iceberg") \
    .getOrCreate()

#Ajuste de log WARN log para ERROR
spark.sparkContext.setLogLevel("ERROR")

In [14]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType, TimestampType, IntegerType

## Orders Events

In [15]:
orders_path = '../Data/order_events.jsonl'

order_schema = StructType([
    StructField("event_id", StringType(),  True),
    StructField("user_id", StringType(),  True),
    StructField("order_id", StringType(),  True),  
    StructField("product_id", StringType(),  True),
    StructField("event_type", StringType(),  True),  
    StructField("price", StringType(),  True),
    StructField("region", StringType(),  True),
    StructField("event_timestamp", StringType(),  True),
    StructField("is_delayed", StringType(),  True),
    StructField("is_returned", StringType(),  True),
])


order_df = (
    spark.read    
    .schema(order_schema)             
    .json(orders_path)
)

In [16]:
order_df.show(5)

+--------------------+-------+------------+----------+-----------------+-------+------+--------------------+----------+-----------+
|            event_id|user_id|    order_id|product_id|       event_type|  price|region|     event_timestamp|is_delayed|is_returned|
+--------------------+-------+------------+----------+-----------------+-------+------+--------------------+----------+-----------+
|20b92918-abdd-454...|   7231|ORD-d4183470|   PRD-098|    ORDER_CREATED|1703.07|    BA|2025-06-13T00:59:...|      NULL|       NULL|
|429dc28f-00e4-467...|   7231|ORD-d4183470|   PRD-098|PAYMENT_CONFIRMED|1703.07|    BA|2025-06-13T01:04:...|      NULL|       NULL|
|560a39fe-6289-4e6...|   7231|ORD-d4183470|   PRD-098|     ORDER_PACKED|1703.07|    BA|2025-06-13T01:59:...|      NULL|       NULL|
|0c71b51e-cd54-48c...|   7231|ORD-d4183470|   PRD-098|    ORDER_SHIPPED|1703.07|    BA|2025-06-13T02:59:...|      NULL|       NULL|
|5affd54a-1522-407...|   7231|ORD-d4183470|   PRD-098|  ORDER_DELIVERED|1703

In [25]:
(
    order_df
    .writeTo("iceberg.bronze.tbl_bronze_order_events")
    .createOrReplace()
)

## Products Catalog

In [26]:
product_catalog_path = '../Data/products.csv'

product_catalog_schema = StructType([
    StructField("product_id", StringType(),  True),
    StructField("product_name", StringType(),  True),
    StructField("category", StringType(),  True),
    StructField("price", StringType(),  True),
])


product_catalog_df = (
    spark.read
    .option("header", True)    
    .option("delimiter", ",")   
    .schema(product_catalog_schema)             
    .csv(product_catalog_path)
)


In [27]:
product_catalog_df.show(5)

+----------+--------------------+--------+-------+
|product_id|        product_name|category|  price|
+----------+--------------------+--------+-------+
|   PRD-001|Flash Drive SanDi...| Storage|2753.42|
|   PRD-002|      HDD Seagate 71| Storage| 2658.0|
|   PRD-003|       HDD Seagate 3| Storage|3723.28|
|   PRD-004|     Gaming Chair 67|  Gaming| 4398.2|
|   PRD-005|     Joystick Xbox 3|  Gaming|2738.95|
+----------+--------------------+--------+-------+
only showing top 5 rows



In [28]:
(
    product_catalog_df
    .writeTo("iceberg.bronze.tbl_bronze_product_catalog")
    .createOrReplace()
)

### Camada Silver

In [29]:
spark.sql("SHOW TABLES in bronze").toPandas()

,namespace,tableName,isTemporary
0,bronze,nyc_taxis,False
1,bronze,tbl_bronze_order_events,False
2,bronze,tbl_bronze_product_catalog,False


In [30]:
spark.sql("select * from iceberg.bronze.tbl_bronze_product_catalog").show(5)

+----------+--------------------+--------+-------+
|product_id|        product_name|category|  price|
+----------+--------------------+--------+-------+
|   PRD-001|Flash Drive SanDi...| Storage|2753.42|
|   PRD-002|      HDD Seagate 71| Storage| 2658.0|
|   PRD-003|       HDD Seagate 3| Storage|3723.28|
|   PRD-004|     Gaming Chair 67|  Gaming| 4398.2|
|   PRD-005|     Joystick Xbox 3|  Gaming|2738.95|
+----------+--------------------+--------+-------+
only showing top 5 rows



In [31]:
spark.sql("""
    CREATE OR REPLACE TABLE iceberg.silver.tbl_silver_product_catalog
    AS
    SELECT
        CAST(product_id AS STRING)          AS product_id,
        CAST(product_name AS STRING)        AS product_name,
        CAST(category AS STRING)            AS category,
        CAST(price AS DOUBLE)               AS price
        
    FROM iceberg.bronze.tbl_bronze_product_catalog

""")

DataFrame[]

In [32]:
spark.table("iceberg.silver.tbl_silver_product_catalog").dtypes

[('product_id', 'string'),
 ('product_name', 'string'),
 ('category', 'string'),
 ('price', 'double')]

In [ ]:
spark.stop()